# ManiSkill 3 — SAPIEN GPU 并行操作仿真一键预览

_One-click preview of ManiSkill 3 — GPU-parallelized manipulation sim on SAPIEN._

基于 [haosulab/ManiSkill](https://github.com/haosulab/ManiSkill)（RSS 2025）。ManiSkill 把仿真**和视觉渲染都搬到 GPU**，单卡 RTX 4090 可达 30,000+ FPS，是本仓 `sapien-experience` 的 SAPIEN 系基准之一（与 [RoboTwin](./RoboTwin.ipynb) 并列）。

_Built on ManiSkill 3 (RSS 2025). It runs **both physics and visual rendering on the GPU** (30,000+ FPS on one RTX 4090). One of the two SAPIEN-based benchmarks in this repo, alongside [RoboTwin](./RoboTwin.ipynb)._

## 四轴一键预览 / Four-axis one-click preview

| 轴 / Axis | 内容 / Coverage | 章节 |
|---|---|---|
| **机器人 / Robot** | 机械臂 · 移动操作 · 人形 · 四足 · 灵巧手 / arms · mobile · humanoid · quadruped · dex hands | §1 |
| **任务场景 / Task** | PickCube · StackCube · PegInsertion · PlugCharger · 画图 · 推 T / drawing · PushT | §2 |
| **求解器 / Solver** | ⭐ 运动规划真解任务（无需训练）/ motion planning solves tasks, no training | §3 |
| **渲染 / Render** | GPU 并行多环境 + 光追 / GPU-parallel multi-env + ray tracing | §4–§5 |

**机制 / Mechanism**：所有逻辑用 ManiSkill 自带 `python -m mani_skill.examples.*` 一等入口，notebook 只一行调用；环境内省走 `scripts/maniskill_demo.py`。

**前置 / Prereqs**：`conda` · NVIDIA GPU + Vulkan · 显示器 `DISPLAY=:0`（GUI 预览用）· ~8 GB 磁盘（env + 资产）。

---
## 0. 安装 conda env `maniskill`（幂等 / idempotent）

**做了什么 / What it does**（见 `scripts/install_maniskill_env.sh`）：
1. `conda create -n maniskill python=3.11`
2. `pip install torch`
3. `pip install -e dependencies/ManiSkill`（pin 住的 submodule）
4. sanity check：`mani_skill` / `sapien` / `torch.cuda` / Vulkan ICD

首次约 5–10 分钟；已装好则秒跳过。

In [ ]:
!bash scripts/install_maniskill_env.sh

In [ ]:
!conda run -n maniskill --no-capture-output python scripts/maniskill_demo.py status

### 0.1 看注册了哪些任务 / 机器人 (introspect)

_List the registered task env ids and robot uids — these are the `-e` / `-r` values used below._

In [ ]:
!conda run -n maniskill --no-capture-output python scripts/maniskill_demo.py list

In [ ]:
!conda run -n maniskill --no-capture-output python scripts/maniskill_demo.py list --robots

---
## 1. 机器人画廊 (Robot Gallery)

ManiSkill 的招牌是**机器人形态极其多样**。`demo_robot` 在 SAPIEN viewer 里加载指定机器人并停在第一个 keyframe，鼠标拖动观察，`Ctrl-C` 退出。

_ManiSkill's headline is embodiment diversity. `demo_robot` loads a robot at its first keyframe in the SAPIEN viewer. Drag to orbit, `Ctrl-C` to quit._

In [ ]:
# 预下载画廊机器人资产(部分机器人 URDF 不随包,首次用需联网下载;-y 跳过确认,幂等)
# Pre-download robot assets (some robots' URDFs aren't bundled). Idempotent, skips existing.
!conda run -n maniskill python - <<'PY'
import subprocess
robots = ['panda', 'xarm6_robotiq', 'so100', 'fetch', 'unitree_g1', 'unitree_h1', 'anymal_c', 'unitree_go2', 'allegro_hand_right', 'trifingerpro']
for r in robots:
    subprocess.run(['python','-m','mani_skill.utils.download_asset', r, '-y'])
PY

In [ ]:
# 🦾 Franka Panda（最常用机械臂） / the canonical 7-DoF arm
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_robot -r panda

In [ ]:
# 🦾 xArm6 + Robotiq 夹爪 / xArm6 with Robotiq gripper
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_robot -r xarm6_robotiq

In [ ]:
# 🦾 SO-100（低成本开源臂） / low-cost open-source arm
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_robot -r so100

In [ ]:
# 🛒 Fetch（移动操作平台） / mobile manipulator
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_robot -r fetch

In [ ]:
# 🧍 Unitree G1（人形） / humanoid
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_robot -r unitree_g1

In [ ]:
# 🧍 Unitree H1(人形)/ H1 humanoid
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_robot -r unitree_h1

In [ ]:
# 🐕 ANYmal-C（四足） / quadruped
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_robot -r anymal_c

In [ ]:
# 🐕 Unitree Go2（四足） / quadruped
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_robot -r unitree_go2

In [ ]:
# ✋ Allegro Hand（16-DoF 灵巧手） / 16-DoF dexterous hand
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_robot -r allegro_hand_right

In [ ]:
# 🤏 TriFinger（三指操作） / tri-finger manipulator
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_robot -r trifingerpro

---
## 2. 任务场景预览 (Task Scene Preview)

`demo_random_action --render-mode human` 加载任务场景并随机采样动作（机器人乱动，**只看场景与初始布局**）。viewer 常驻，`Ctrl-C` 退出。想看机器人**真的把任务做出来**，去 §3 运动规划。

_Loads the task scene and samples random actions (robot flails — **scene preview only**). Viewer stays open; `Ctrl-C` to quit. For the robot actually solving the task, see §3._

In [ ]:
# 📦 抓方块 / Pick cube
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_random_action -e PickCube-v1 --render-mode human

In [ ]:
# 🧱 叠方块 / Stack cube
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_random_action -e StackCube-v1 --render-mode human

In [ ]:
# 🔩 侧插销 / Peg insertion
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_random_action -e PegInsertionSide-v1 --render-mode human

In [ ]:
# 🔌 插充电器 / Plug charger
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_random_action -e PlugCharger-v1 --render-mode human

In [ ]:
# 🔤 推 T 形 / Push-T
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_random_action -e PushT-v1 --render-mode human

In [ ]:
# ⚽ 滚球 / Roll ball
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_random_action -e RollBall-v1 --render-mode human

In [ ]:
# 👉 杆捅方块 / Poke cube
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_random_action -e PokeCube-v1 --render-mode human

In [ ]:
# 📐 画三角 / Draw triangle
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_random_action -e DrawTriangle-v1 --render-mode human

In [ ]:
# 🤝 双臂协作抓方块 / Two-arm pick
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_random_action -e TwoRobotPickCube-v1 --render-mode human

---
## 3. ⭐ 运动规划真解任务 (Motion Planning — Solve, No Training)

这是 ManiSkill 最爽的一键演示：`mplib` 运动规划器**不需要任何训练**就把任务做出来。`panda.run --vis` 在 viewer 里实时展示 Panda 一步步抓取、对位、插入。

_The most satisfying one-click demo: the `mplib` motion planner **solves the task with zero training**. `panda.run --vis` shows Panda grasping / aligning / inserting live in the viewer._

> 💡 已写好运动规划解的任务 / Tasks with a written solver：`PickCube · PushCube · StackCube · PegInsertionSide · PlugCharger · PullCube · PullCubeTool · LiftPegUpright · PlaceSphere · RollBall · DrawTriangle · DrawSVG · StackPyramid`.

In [ ]:
# 📦 抓方块 / Pick cube — 运动规划实解 / motion-planning solve, live viewer
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.motionplanning.panda.run -e PickCube-v1 -n 3 --vis

In [ ]:
# 🧱 叠方块 / Stack cube — 运动规划实解 / motion-planning solve, live viewer
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.motionplanning.panda.run -e StackCube-v1 -n 3 --vis

In [ ]:
# 🔩 侧插销（精密对位）/ Peg insertion (precise) — 运动规划实解 / motion-planning solve, live viewer
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.motionplanning.panda.run -e PegInsertionSide-v1 -n 3 --vis

In [ ]:
# 🔌 插充电器 / Plug charger — 运动规划实解 / motion-planning solve, live viewer
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.motionplanning.panda.run -e PlugCharger-v1 -n 3 --vis

In [ ]:
# 🔺 叠金字塔 / Stack pyramid — 运动规划实解 / motion-planning solve, live viewer
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.motionplanning.panda.run -e StackPyramid-v1 -n 3 --vis

### 3.1 无头出视频 / Headless → MP4 (no DISPLAY needed)

不开窗口、批量生成成功轨迹并录像（`--shader rt` 光追渲染）。视频落在 `demos/<env>/`。

_Headless: generate successful trajectories + ray-traced videos. Output under `demos/<env>/`._

In [ ]:
# PegInsertionSide ×5 成功轨迹 + 光追视频 / 5 successful trajs + RT videos
!conda run -n maniskill --no-capture-output python -m mani_skill.examples.motionplanning.panda.run \
    -e PegInsertionSide-v1 -n 5 --only-count-success --save-video --shader rt --record-dir demos

---
## 4. GPU 并行多环境 (GPU-Parallel Multi-Env)

ManiSkill 的核心卖点：**几百上千个环境同时在一张卡上跑**。`-n 64 --render-mode human` 把 64 个并行 env 画在同一个 viewer 里（`parallel_in_single_scene`），直观感受 GPU 并行仿真。

_The core selling point: hundreds of envs on one GPU. `-n 64 --render-mode human` renders 64 parallel envs in one viewer._

In [ ]:
# 64 个 PickCube 并行画在一个场景里 / 64 parallel PickCube envs in one scene
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_random_action -e PickCube-v1 -n 64 -b gpu --render-mode human

### 4.1 吞吐 benchmark / Throughput

无渲染纯仿真吞吐（state-only），看 GPU 并行的 FPS 量级。

_Headless state-only throughput — the GPU-parallel FPS headline._

In [ ]:
# 1024 个并行 env 的 state-only 吞吐 / state-only throughput, 1024 parallel envs
!conda run -n maniskill --no-capture-output python -m mani_skill.examples.benchmarking.gpu_sim -e PickCube-v1 -n 1024

---
## 5. 光线追踪渲染 (Ray-Traced Rendering)

`--shader rt` 切换到路径追踪渲染器，照片级画质（慢但好看）；`rt-fast` 是快速低质版。

_`--shader rt` switches to the path-tracing renderer for photorealistic frames (`rt-fast` = faster/lower-quality)._

In [ ]:
# 光追 PickCube，单环境 viewer / ray-traced PickCube in viewer
!DISPLAY=:0 conda run -n maniskill --no-capture-output python -m mani_skill.examples.demo_random_action -e PickCube-v1 --shader rt --render-mode human

---
## 6. 下载并回放官方示范 (Demonstrations: Download & Replay)

ManiSkill 在 HF [`haosulab/ManiSkill_Demonstrations`](https://huggingface.co/datasets/haosulab/ManiSkill_Demonstrations) 提供示范轨迹。下载后用 `replay_trajectory` 回放并出光追视频。

_Download official demo trajectories from HF, replay with `replay_trajectory` into ray-traced videos._

In [ ]:
# 下载 PickCube 示范 / download PickCube demos (~tens of MB)
!conda run -n maniskill --no-capture-output python -m mani_skill.utils.download_demo PickCube-v1

In [ ]:
# 回放 + 出光追视频 / replay + ray-traced video (video next to the .h5)
!conda run -n maniskill --no-capture-output bash -lc 'H5=$(find ~/.maniskill/demos -path "*PickCube-v1*" -name "*.h5" | head -1); \
  echo "replaying $H5"; \
  python -m mani_skill.trajectory.replay_trajectory --traj-path "$H5" \
    --save-video --render-mode rgb_array --shader rt -o none --count 5'

---
## 7. 进阶：RL / 模仿学习 baselines (Advanced)

ManiSkill 自带 PPO / SAC / TD-MPC2 / BC / Diffusion Policy / ACT 等 baseline，在 `dependencies/ManiSkill/examples/baselines/`。训练耗时，未纳入一键预览。

_Built-in RL/IL baselines (PPO / SAC / TD-MPC2 / BC / Diffusion Policy / ACT) live in `examples/baselines/`. Training is time-consuming — out of scope for this preview notebook._

```bash
# 例：PPO 训 PickCube（state）/ e.g. PPO on PickCube
conda run -n maniskill python dependencies/ManiSkill/examples/baselines/ppo/ppo.py \
    --env_id PickCube-v1 --num_envs 1024 --total_timesteps 10_000_000
```

## 进一步 / Next steps
- **更多任务资产**：`python -m mani_skill.utils.download_asset PickSingleYCB-v1`（YCB 真实物体）
- **真实物体拣选**：`PickSingleYCB-v1` / `PickClutterYCB-v1`（需先下资产）
- **Real2Sim / Sim2Real**：见官方 docs <https://maniskill.readthedocs.io>
- **文档**：<https://maniskill.readthedocs.io/en/latest/user_guide/getting_started/quickstart.html>